# Classiq logo oracle — verified depth 258

This companion preserves the original baseline notebook. The new standalone oracle is **258 depth / 1188 CX / 18 qubits**, versus the previous verified local depth 456. The 183-depth benchmark remains unfinished.

The construction keeps the two-level-comparison identity, distributes lookup phases over temporary coordinate/output parities, carries Gray-code offsets between sweeps, and uses a factored 13-layer phase kernel. See [the derivation](docs/DISTRIBUTED_LOOKUP_258.md).

Use the workspace `.venv` Python environment. These cells inspect the protected package and rebuild a fresh candidate under a new temporary directory; they do not submit or run cloud synthesis.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import tempfile

ROOT = Path.cwd()
assert (ROOT / "src/build_distributed_best.py").exists(), "Open this notebook from the classiq workspace root"
sys.path.insert(0, str(ROOT / "src"))
from qiskit import qasm2
from build_distributed_best import build
from exhaustive_verify import exhaustive

PACKAGE = ROOT / "artifacts/258"
QASM = PACKAGE / "distributed_level_258.qasm"


## Check the authoritative file

The QASM is the scored oracle. Coordinates occupy q[0:12]; only q[12:18] starts clean. The report checks every coordinate input with one shared global phase.


In [2]:
circuit = qasm2.load(QASM)
report = json.loads(QASM.with_suffix(".exhaustive.json").read_text())
sha = hashlib.sha256(QASM.read_bytes()).hexdigest()
assert sha == report["sha256"]
assert report["basis_inputs_checked"] == 4096
assert set(circuit.count_ops()) <= {"u3", "cx"}
print({"depth": circuit.depth(), "CX": circuit.count_ops()["cx"], "width": circuit.num_qubits})
print("SHA-256:", sha)
print("Maximum exhaustive error:", report["max_error"])


{'depth': 258, 'CX': 1188, 'width': 18}
SHA-256: b2a2e8ac6a6d7ee2c2e4ec11bcca4b4ba4fe54aab15b11c71236efcecdca3066
Maximum exhaustive error: 1.2959208739370123e-14


## What changed

A Walsh lookup phase for target `t` and coordinate mask `s` can be applied to the parity `t XOR parity(s, coordinate)`. The old implementation kept these parities on output ancillas. The new implementation temporarily places them on coordinate wires too, evaluates the phases in parallel, and restores the complete linear basis.

Three coordinate bits and three outputs form six phase hosts. Four structured host bases cover the required high-coordinate parities. Each transition costs three CX layers. The low-coordinate Gray walks need not return to zero between bases; paired offsets cancel during coordinate restoration.

The two phase applications still implement

`logo(x,y) = [u1(y)+v1(x)>=6] XOR [u2(y)+v2(x)>=6]`.

The kernel polynomial factors as

`a*f XOR (a XOR f)*(b*d XOR c*e)`.

The saved 13-layer kernel implements this exactly on all 64 code states, up to one shared global phase.


In [3]:
recipe = json.loads((PACKAGE / "recipe.json").read_text())
kernel = qasm2.load(PACKAGE / "kernel.qasm")
print("Kernel:", kernel.depth(), "depth;", kernel.count_ops()["cx"], "CX")
print("y code subsets:", recipe["y"]["triple"])
print("x code subsets:", recipe["x"]["triple"])


Kernel: 13 depth; 15 CX
y code subsets: [56, 52, 38]
x code subsets: [20, 34, 56]


## Rebuild and verify a fresh candidate

This is a deterministic replay, not a search. Component verification checks the full nine-qubit operator, including arbitrary output values. The final exhaustive check uses the exact newly serialized standalone QASM.


In [4]:
rebuilt, component_checks = build(recipe, kernel, verify_components=True)
NEW_DIR = Path(tempfile.mkdtemp(prefix="classiq_distributed_replay_"))
NEW_QASM = NEW_DIR / f"distributed_level_{rebuilt.depth()}.qasm"
NEW_QASM.write_text(qasm2.dumps(rebuilt))
assert hashlib.sha256(NEW_QASM.read_bytes()).hexdigest() == sha
print("Exact replay:", NEW_QASM)
print("Maximum component error:", max(item["error"] for item in component_checks))
exhaustive(NEW_QASM)


Exact replay: /var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/classiq_distributed_replay_2zk10zyl/distributed_level_258.qasm
Maximum component error: 3.1086778004677257e-15
gate 0 / 2274 support 2
gate 250 / 2274 support 8
gate 500 / 2274 support 8
gate 750 / 2274 support 8
gate 1000 / 2274 support 32
gate 1250 / 2274 support 8
gate 1500 / 2274 support 1
gate 1750 / 2274 support 32
gate 2000 / 2274 support 8
gate 2250 / 2274 support 8
{
  "qasm": "/private/var/folders/tm/6bh1bn3x6pgfgp8nvpknylq40000gn/T/classiq_distributed_replay_2zk10zyl/distributed_level_258.qasm",
  "sha256": "b2a2e8ac6a6d7ee2c2e4ec11bcca4b4ba4fe54aab15b11c71236efcecdca3066",
  "basis_inputs_checked": 4096,
  "width": 18,
  "depth": 258,
  "cx_count": 1188,
  "max_error": 1.2959208739370123e-14,
  "ancilla_error": 0.0,
  "discarded_amplitude_bound": 2.3260798255352838e-14,
  "peak_sparse_support": 32,
  "elapsed_seconds": 0.46553802490234375,
  "verification": "Exhaustive numerical verification of all 4096 clean-an

## Matching QMOD and remaining work

`artifacts/258/distributed_level_258.qmod` contains a gate-for-gate matching oracle function. Its `main` adds twelve Hadamards only as the synthesis harness; those gates are absent from the scored QASM. The gate order, wire indices, and parameters were compared against the QASM with zero parameter difference.

The current result reduces depth by 43.4% from 456. The three 77–78-layer lookup stages remain the main cost. Reaching 183 requires further work; neither this notebook nor the package claims a leaderboard submission or rank one.
